In [0]:
from pydeequ.checks import Check,CheckLevel
from pydeequ.verification import VerificationSuite,VerificationResult
from pyspark.sql import functions as F
import datetime as _dt

In [0]:
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "deafult"



In [0]:

spark.sql(f"create schema if not exists {catalog}.ops")
spark.sql(f"""create table if not exists {catalog}.ops.dq_results(
    business_date Date,
    dataset string,
    check_name string,
    status string,
    constrant string,
    message string,
    recorded_at timestamp
) using delta""")


src = spark.table(f"{catalog}.bronze.customers_inc") \
           .where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

In [0]:
src.display()

In [0]:
check = (Check(spark,CheckLevel.Error,"Customer Data Check")
        .hasSize(lambda x: x > 0)
        .isComplete("customer_name")
        .isComplete("customer_address")
        .isComplete("email"))

result = (VerificationSuite(spark).onData(src).addCheck(check).run())
df = VerificationResult.checkResultsAsDataFrame(spark,result)
display(df)


In [0]:
out = (df.withColumn("business_date",F.to_date(F.lit(arrival_date))) \
         .withColumn("dataset",F.lit("bookings_inc")) \
         .withColumn("recorded_at",F.current_timestamp()))
out.select("business_date","dataset","check","check_status","constraint","constraint_status","constraint_message","recorded_at") \
    .write.mode("append").option("mergeSchema",True).saveAsTable(f"{catalog}.ops.dq_results")

In [0]:
if result.status != "Success":
    raise Exception("Data Quality Check Failed")

print("Customer DQ Passed")